In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from langchain.agents import create_agent
from langchain_ollama import chat_models, ChatOllama
from langchain.tools import tool
import sqlite3

conn = sqlite3.connect("store.db")
cursor = conn.cursor()
columns = cursor.execute("select name from sqlite_master where type='table'").fetchall()


def fetch_product_rating(id):
    cursor.execute("SELECT product_id, avg(rating) as avg_rating, count(*) as review_count FROM reviews WHERE product_id = ? group by product_id", (id,))
    row = cursor.fetchall()[0]
    if row:
        avg_rating = round(row[1],2) if row[1] else 0.0
        review_count = row[2]  if row[2] else 0
    else:
        avg_rating = 0.0
        review_count = 0

    dic = {"product_id": id, "avg_rating": avg_rating, "review_count": review_count}
    return dic

def ratings_for_different_products(ids):
    ratings = []
    for id in ids:
        ratings.append(fetch_product_rating(id))
    return ratings

In [5]:
def search_products(query, max_price=None, is_organic=None):
    sql_query = "select name, category, description, price, is_organic from products where 1=1"
    all_parameters = []

    if query:
        sql_query += " AND (name LIKE ? OR category LIKE ? OR description LIKE ?)"
        query_param = f"%{query}%"
        all_parameters.extend([query_param,query_param,query_param])

    if max_price is not None:
        sql_query += " AND price <= ?"
        all_parameters.append(max_price)

    if is_organic is not None:
        sql_query += " AND organic = ?"
        all_parameters.append(is_organic)

    cursor.execute(sql_query, all_parameters)
    all_rows = cursor.fetchall()

    result_list = []

    for i in all_rows:
        result_dict = {
            "name" : i[0],
            "category" : i[1],
            "description" : i[2] if i[2] else "",
            "price" : i[3] if i[3] else 0.0,
            "is_organic" : bool(i[4])
        }
        result_list.append(result_dict)

    return result_list

In [6]:
def get_rating(id):
    product_rating_data = fetch_product_rating(id)
    return product_rating_data

In [7]:
def search_products(name, category, description="", max_price=None, is_organic=None):
    sql_query = "select name, category, description, price, is_organic from products where 1=1"
    all_parameters = []

    sql_query += " AND (name LIKE ? OR category LIKE ? OR description LIKE ?)"
    all_parameters.extend([name,category,description])

    if max_price is not None:
        sql_query += " AND price <= ?"
        all_parameters.append(max_price)

    if is_organic is not None:
        sql_query += " AND organic = ?"
        all_parameters.append(is_organic)

    cursor.execute(sql_query, all_parameters)
    all_rows = cursor.fetchall()

    result_list = []

    for i in all_rows:
        result_dict = {
            "name" : i[0],
            "category" : i[1],
            "description" : i[2] if i[2] else "",
            "price" : i[3] if i[3] else 0.0,
            "is_organic" : bool(i[4])
        }
        result_list.append(result_dict)

    return result_list

In [8]:
lk = search_products("Organic Raw Honey","honey")
lk

[{'name': 'Organic Raw Honey',
  'category': 'honey',
  'description': 'Pure organic raw honey, unfiltered and cold-pressed',
  'price': 14.99,
  'is_organic': True},
 {'name': 'Wildflower Honey',
  'category': 'honey',
  'description': 'Natural wildflower honey from local beekeepers',
  'price': 12.99,
  'is_organic': False},
 {'name': 'Organic Manuka Honey',
  'category': 'honey',
  'description': 'Premium organic Manuka honey from New Zealand',
  'price': 29.99,
  'is_organic': True},
 {'name': 'Clover Honey',
  'category': 'honey',
  'description': 'Classic clover honey, smooth and sweet',
  'price': 8.99,
  'is_organic': False},
 {'name': 'Organic Buckwheat Honey',
  'category': 'honey',
  'description': 'Dark and robust organic buckwheat honey, antioxidant-rich',
  'price': 18.99,
  'is_organic': True},
 {'name': 'Orange Blossom Honey',
  'category': 'honey',
  'description': 'Light and floral orange blossom honey',
  'price': 15.99,
  'is_organic': False},
 {'name': 'Organic Aca